## 传递数据

RunnablePassthrough 允许不改变或添加额外的键来传递输入。这通常与 RunnableParallel 结合使用，将数据分配给映射中的新键。

RunnablePassthrough() 单独调用，将简单地获取输入并将其传递。

使用分配 ( RunnablePassthrough.assign(...) ) 调用的 RunnablePassthrough 将获取输入，并将添加传递给分配函数的额外参数。

In [12]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

如上所示， passed 键是用 RunnablePassthrough() 调用的，因此它只是传递 {'num': 1} 。

在第二行中，我们使用 RunnablePastshrough.assign 和一个将数值乘以 3 的 lambda。在这种情况下， extra 设置为 {'num': 1, 'mult': 3} ，这是原始的添加了 mult 键的值。

最后，我们还在映射中使用 modified 设置了第三个键，它使用 lambda 来设置单个值，在 num 上加 1，从而得到 modified 键，其值为 < b2> 。

## 运行自定义函数

您可以在管道中使用任意函数。

请注意，这些函数的所有输入都必须是单个参数。如果您有一个接受多个参数的函数，则应该编写一个接受单个输入并将其解包为多个参数的包装器。



In [13]:
from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)

In [14]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI


def length_function(text):
    return len(text)


def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)


def multiple_length_function(_dict):
    return _multiple_length_function(_dict["text1"], _dict["text2"])


prompt = ChatPromptTemplate.from_template("what is {a} + {b}")


chain1 = prompt | model

chain = (
    {
        "a": itemgetter("foo") | RunnableLambda(length_function),
        "b": {"text1": itemgetter("foo"), "text2": itemgetter("bar")} | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
)

In [15]:
chain.invoke({"foo": "bar", "bar": "gah"})

AIMessage(content="<think>\nOkay, so I need to figure out what 3 plus 9 equals. Let me think. First, addition is when you combine two numbers. So adding 3 and 9 should be straightforward.\n\nWait, let me make sure I remember the rules correctly. When you add two numbers, you just add their values. So 3 plus 9... Hmm, 3 plus 5 would be 8, right? Then if I add another 4 to that, it's 12. But wait, is there a different way to do this?\n\nAlternatively, maybe breaking down the numbers. Let me try adding them in parts. 3 plus 9 can be thought of as 3 + 8 + 1. So 3 + 8 is 11, and then 11 + 1 equals 12. That seems correct.\n\nAnother way to check: if I add 3 and 9 directly, it's just 12. Because 3 plus 9 is the same as 9 plus 3. So both methods should give me the same answer. \n\nWait, maybe I can visualize this with numbers. Imagine a number line. Starting at 0, adding 3 gets me to 3. Then adding another 9 would take me to 12. Yep, that makes sense.\n\nIs there any chance I might have made a

## 根据输入动态路由逻辑

路由允许您创建非确定性链，其中上一步的输出定义下一步。路由有助于提供与 LLMs 交互的结构和一致性。

执行路由的方法有两种：

1. 有条件地从 RunnableLambda 返回可运行对象（推荐）
2. 使用 RunnableBranch 

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
        """鉴于下面的用户问题，将其分类为“LangChain”、“OpenAI”或“其他”。

不要用超过一个字来回应。

<question>
{question}
</question>

分类："""
)

chain = (
    prompt
    | model
    | StrOutputParser()
)

chain.invoke({"question": "how do I call OpenAI?"})

'<think>\n嗯，用户问的是如何调用OpenAI。首先，我需要确定用户的问题是否与他们所使用的工具相关。问题中提到的关键词是“OpenAI”，而用户可能是在询问如何使用这个平台来实现某个功能。\n\n接下来，我要考虑用户的具体需求。如果用户想了解如何调用OpenAI的API，那么正确的分类应该是“LangChain”或者“OpenAI”。但根据问题本身，用户并没有提到他们使用的是LangChain或者其他工具，而是直接问如何调用OpenAI。因此，可能需要进一步确认。\n\n不过，根据常规情况，当用户询问如何调用某个特定平台时，正确的分类通常是该平台的名称。所以在这里，用户的问题涉及的是OpenAI，而问题本身并未说明是否与LangChain相关。因此，正确的分类应该是“OpenAI”，因为这是直接提到的工具。\n</think>\n\nOpenAI'

In [17]:
langchainPrompt = PromptTemplate.from_template(
        """您是 langchain 方面的专家。 
回答问题时始终以“正如老陈告诉我的那样”开头。 
回答以下问题：

问题：{question}
回答："""
)
langchain_chain = langchainPrompt | model

In [18]:
OpenAIPrompt = PromptTemplate.from_template(
        """您是 OpenAI 方面的专家。 
回答问题时始终以“正如奥特曼告诉我的那样”开头。 
回答以下问题：

问题：{question}
回答："""
)
OpenAI_chain = OpenAIPrompt | model

In [19]:
generalPrompt = PromptTemplate.from_template(
        """ 回答以下问题：

问题：{question}
回答："""
)
general_chain = generalPrompt | model

In [20]:
def route(info):
    if "OpenAI" in info["topic"]:
        return OpenAI_chain
    elif "LangChain" in info["topic"]:
        return langchain_chain
    else:
        return general_chain

In [21]:
from langchain_core.runnables import RunnableLambda

full_chain = {"topic": chain, "question": lambda x: x["question"]} | RunnableLambda(route)

In [22]:
full_chain.invoke({"question": "我如何使用OpenAI的模型?"})

AIMessage(content='<think>\n好的，用户问的是如何使用OpenAI的模型。首先，我需要确保回答符合要求，即从“正如奥特曼告诉我的那样”开头，并且以正确的格式呈现。\n\n接下来，我要回忆OpenAI的主要模型和相关技术。比如，GPT系列、Hugging Face Transformers等。然后，考虑用户可能的需求，他们可能想知道具体的步骤或配置方法。需要确保回答清晰易懂，涵盖基础使用方式，同时保持简洁。\n\n还要注意用户可能的疑问点，比如安装指南、API调用示例、模型参数调整等。但根据问题本身，用户只需要知道如何使用模型，所以重点放在指导步骤上。最后检查是否符合所有要求，包括开头和格式。\n</think>\n\n正如奥特曼告诉我的那样，使用OpenAI的模型需要先了解其核心功能和技术支持。首先，您可以通过访问Hugging Face的官方平台下载并安装适合您的模型（如GPT-3或GPT-4）。接下来，根据具体需求配置API密钥，并通过调用预训练模型的接口进行数据处理和推理任务。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 230, 'prompt_tokens': 51, 'total_tokens': 281, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen3-0.6b', 'system_fingerprint': 'qwen3-0.6b', 'id': 'chatcmpl-ev4zq7d4w8o4rzyvmq8hlj', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--4c5e6ecf-4f10-479f-b842-b140644c8e5a-0', usage_metadata={'input_tokens': 51, 'output_tokens': 230, 'total_tokens': 281, 'input_token_details': {}, 'o